In [1]:
import geopandas as gpd
import pandas as pd

In [ ]:
# from shapely.geometry import Point

# Primary Care Hospitals
favelas = gpd.read_file("/Volumes/ssd/Doutorado/DATASET/IBGE/Favelas e Comunidades Urbanas - 2022/poligonos/fcu_blank/qg_2022_670_fcu_agreg.shp")
primary_care_hospitals = gpd.read_file("/Volumes/ssd/Doutorado/DATASET/IBGE/base_grafica AGSN 2019/ESaude_Atencao_Primaria/ESaude_Atencao_Primaria.shp")
inpatient_hospitals = gpd.read_file("/Volumes/ssd/Doutorado/DATASET/IBGE/base_grafica AGSN 2019/ESaude_Internacao_Observacao/ESaude_Internacao_Observacao.shp")

# Ensure both datasets are in the same coordinate system
favelas = favelas.to_crs(epsg=3857)  # Example: metric (meters)
primary_care_hospitals = primary_care_hospitals.to_crs(favelas.crs)
inpatient_hospitals = inpatient_hospitals.to_crs(favelas.crs)

# Create list of hospital geometries
primary_care_hospital_points = primary_care_hospitals.geometry.tolist()
inpatient_hospital_points = inpatient_hospitals.geometry.tolist()

In [ ]:
def calculate_distances(polygon):
    centroide = polygon.centroid
    d1 = 0 if any(polygon.contains(p) for p in primary_care_hospital_points) else min(centroide.distance(p) for p in primary_care_hospital_points)
    d2 = 0 if any(polygon.contains(p) for p in inpatient_hospital_points) else min(centroide.distance(p) for p in inpatient_hospital_points)
    return pd.Series({'primary_care_hospital': d1, 'inpatient_hospital': d2})

In [ ]:
favelas[['primary_care_hospital', 'inpatient_hospital']] = favelas.geometry.apply(calculate_distances)

In [6]:
favelas.to_file('/Volumes/ssd/Doutorado/DATASET/IBGE/base_grafica AGSN 2019/DIST_HOSPITALS/dist_hospitals_2019.shp')


/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_10958/1410689187.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  favelas.to_file('/Volumes/ssd/Doutorado/DATASET/IBGE/base_grafica AGSN 2019/DIST_HOSPITALS/dist_hospitals_2019.shp')
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'primary_care_hospital' to 'primary_ca'
  ogr_write(
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'inpatient_hospital' to 'inpatient_'
  ogr_write(
